In [ ]:
# ============================================
# SPY vs Copper vs LQD
# 20-Year Correlation & Risk Analysis (FIXED)
# ============================================

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 6)

# ------------------------------------------------
# 1. 데이터 다운로드 (auto_adjust=True 대응)
# ------------------------------------------------
tickers = {
    "SPY": "SPY",
    "COPPER": "HG=F",   # COMEX Copper Futures
    "LQD": "LQD"
}

start_date = "2005-01-01"

raw = yf.download(
    list(tickers.values()),
    start=start_date,
    auto_adjust=True,
    progress=False
)

# Close 가격만 사용 (auto_adjust 적용된 가격)
data = raw["Close"].copy()
data.columns = tickers.keys()
data = data.dropna()

# ------------------------------------------------
# 2. 로그 수익률
# ------------------------------------------------
log_ret = np.log(data / data.shift(1)).dropna()

# ------------------------------------------------
# 3. 상관관계
# ------------------------------------------------
print("=== Pearson Correlation ===")
display(log_ret.corr(method="pearson"))

print("=== Spearman Correlation ===")
display(log_ret.corr(method="spearman"))

# ------------------------------------------------
# 4. Rolling Correlation (12개월)
# ------------------------------------------------
window = 252

plt.figure()
plt.plot(
    log_ret["SPY"].rolling(window).corr(log_ret["COPPER"]),
    label="SPY vs Copper"
)
plt.plot(
    log_ret["SPY"].rolling(window).corr(log_ret["LQD"]),
    label="SPY vs LQD"
)
plt.axhline(0, linestyle="--", alpha=0.5)
plt.title("Rolling 12M Correlation with SPY")
plt.legend()
plt.grid(True)
plt.show()

# ------------------------------------------------
# 5. 성과 / 리스크 지표
# ------------------------------------------------
def metrics(price):
    ret = price.pct_change().dropna()
    cagr = (price.iloc[-1] / price.iloc[0]) ** (252 / len(ret)) - 1
    vol = ret.std() * np.sqrt(252)

    cum = (1 + ret).cumprod()
    mdd = (cum / cum.cummax() - 1).min()

    return pd.Series({
        "CAGR": cagr,
        "Volatility": vol,
        "MDD": mdd
    })

summary = pd.concat(
    {k: metrics(data[k]) for k in data.columns},
    axis=1
).T

print("=== Performance & Risk Metrics (20Y) ===")
display(summary)

# ------------------------------------------------
# 6. 정규화 가격 비교
# ------------------------------------------------
(data / data.iloc[0] * 100).plot(title="Normalized Price (Start = 100)")
plt.grid(True)
plt.show()
